# 🏥 Multilingual Medical OCR System

> **End-to-end pipeline:** Image → OCR → Translation → BERT NER → PDF Report

| Component | Technology |
|-----------|------------|
| OCR | TrOCR + EasyOCR + Tesseract |
| NLP / NER | `dslim/bert-base-NER` |
| Translation | Google Translate via `deep-translator` |
| PDF | ReportLab |
| Language Detection | `langdetect` |

**⚡ Recommended:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — INSTALL DEPENDENCIES                              ║
# ╚══════════════════════════════════════════════════════════════╝

print('📦 Installing system packages ...')
import subprocess, sys

# System OCR engine
subprocess.run(['apt-get', 'install', '-y', '-q',
    'tesseract-ocr',
    'tesseract-ocr-hin',   # Hindi
    'tesseract-ocr-ben',   # Bengali
    'tesseract-ocr-ori',   # Odia
    'tesseract-ocr-tam',   # Tamil
    'tesseract-ocr-tel',   # Telugu
    'libgl1-mesa-glx',
], check=True, stdout=subprocess.DEVNULL)

print('📦 Installing Python packages (this may take 2-3 minutes) ...')
packages = [
    'transformers>=4.38.0',
    'accelerate>=0.27.0',
    'easyocr>=1.7.1',
    'pytesseract>=0.3.10',
    'opencv-python-headless>=4.9.0',
    'deep-translator>=1.11.4',
    'langdetect>=1.0.9',
    'reportlab>=4.1.0',
    'sentencepiece>=0.1.99',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

print('✅ All dependencies installed successfully!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — IMPORT LIBRARIES                                  ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Standard library ─────────────────────────────────────────────────────────
import os, re, io, time, logging, warnings
from pathlib import Path
from datetime import datetime
from IPython.display import display, Image as IPImage, FileLink
import json

# ── Scientific / Image ────────────────────────────────────────────────────────
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── OCR ───────────────────────────────────────────────────────────────────────
import easyocr
import pytesseract

# ── HuggingFace ───────────────────────────────────────────────────────────────
import torch
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    pipeline as hf_pipeline,
)

# ── Translation ───────────────────────────────────────────────────────────────
from langdetect import detect, LangDetectException
from deep_translator import GoogleTranslator

# ── PDF ───────────────────────────────────────────────────────────────────────
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer,
    Table, TableStyle, HRFlowable,
)
from reportlab.lib.enums import TA_CENTER, TA_LEFT

# ── Google Colab file utils ───────────────────────────────────────────────────
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.WARNING)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Libraries imported | Device: {DEVICE} | PyTorch: {torch.__version__}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — LOAD MODELS (TrOCR + EasyOCR + BERT NER)         ║
# ╚══════════════════════════════════════════════════════════════╝

print('🧠 Loading TrOCR (microsoft/trocr-base-printed) ...')
trocr_processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-printed')
trocr_model = VisionEncoderDecoderModel.from_pretrained(
    'microsoft/trocr-base-printed'
).to(DEVICE)
trocr_model.eval()
print('  ✅ TrOCR loaded')

print('📖 Loading EasyOCR (en + hi) ...')
easy_reader = easyocr.Reader(['en', 'hi'], gpu=(DEVICE == 'cuda'))
print('  ✅ EasyOCR loaded')

print('🤖 Loading BERT NER (dslim/bert-base-NER) ...')
ner_pipeline = hf_pipeline(
    'ner',
    model='dslim/bert-base-NER',
    aggregation_strategy='simple',
    device=0 if DEVICE == 'cuda' else -1,
)
print('  ✅ BERT NER loaded')

print(f'\n🎉 All models loaded on {DEVICE.upper()}!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — IMAGE UPLOAD                                      ║
# ╚══════════════════════════════════════════════════════════════╝

if IN_COLAB:
    print('📂 Please upload a medical image (prescription / lab report / X-ray report):')
    uploaded = colab_files.upload()
    if uploaded:
        IMAGE_PATH = list(uploaded.keys())[0]
        print(f'\n✅ Uploaded: {IMAGE_PATH}')
    else:
        raise ValueError('No file uploaded.')
else:
    # Local use — set your image path here
    IMAGE_PATH = 'sample_prescription.jpg'
    if not os.path.exists(IMAGE_PATH):
        # Create a sample test image with text
        from PIL import ImageDraw, ImageFont
        img = Image.new('RGB', (600, 400), color='white')
        draw = ImageDraw.Draw(img)
        text = (
            'Patient: John Doe\n'
            'DOB: 12/05/1985  Date: 10/04/2024\n'
            'Dr. Smith  |  City Hospital\n\n'
            'Rx:\n'
            'Tab Amoxicillin 500mg  BD x 7 days\n'
            'Tab Paracetamol 650mg  TDS PRN\n'
            'Syr Ibuprofen 200mg/5ml  1 tsp TDS\n\n'
            'Diagnosis: Acute pharyngitis, fever\n'
            'Refills: 0'
        )
        draw.multiline_text((30, 30), text, fill='black', spacing=8)
        img.save(IMAGE_PATH)
        print(f'⚠️ Sample image created: {IMAGE_PATH}')
    else:
        print(f'✅ Using local image: {IMAGE_PATH}')

# Display the uploaded image
orig_img = Image.open(IMAGE_PATH)
plt.figure(figsize=(8, 6))
plt.imshow(orig_img)
plt.title(f'Original Image: {IMAGE_PATH}  ({orig_img.size[0]}×{orig_img.size[1]})',
          fontsize=12, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — IMAGE PREPROCESSING (CLAHE + DENOISE + DESKEW)   ║
# ╚══════════════════════════════════════════════════════════════╝

def preprocess_image(image_input):
    """
    Full image preprocessing pipeline:
    1. Upscale if too small
    2. Convert to grayscale
    3. CLAHE contrast enhancement
    4. Gaussian denoising
    5. Sharpening
    6. Deskew via Hough transform
    Returns (processed_gray: np.ndarray, pil_image: PIL.Image)
    """
    # Load
    if isinstance(image_input, (str, Path)):
        img = cv2.imread(str(image_input))
    elif isinstance(image_input, Image.Image):
        img = cv2.cvtColor(np.array(image_input.convert('RGB')), cv2.COLOR_RGB2BGR)
    else:
        img = image_input.copy()

    h, w = img.shape[:2]
    print(f'  Original size: {w}×{h}')

    # 1. Upscale
    if max(h, w) < 1000:
        scale = 1500 / max(h, w)
        img = cv2.resize(img, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
        print(f'  Upscaled by {scale:.1f}x → {img.shape[1]}×{img.shape[0]}')

    # 2. Grayscale
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 3. CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # 4. Gaussian blur (denoise)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    # 5. Sharpen
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    gray = cv2.filter2D(gray, -1, kernel)

    # 6. Deskew
    try:
        edges = cv2.Canny(gray, 50, 150, apertureSize=3)
        lines = cv2.HoughLines(edges, 1, np.pi / 180, 200)
        if lines is not None:
            angles = []
            for line in lines[:20]:
                rho, theta = line[0]
                if theta < np.pi / 4 or theta > 3 * np.pi / 4:
                    angles.append(theta - np.pi / 2)
            if angles:
                angle_deg = np.median(angles) * 180 / np.pi
                if abs(angle_deg) > 0.5:
                    (hh, ww) = gray.shape
                    M = cv2.getRotationMatrix2D((ww//2, hh//2), angle_deg, 1.0)
                    gray = cv2.warpAffine(gray, M, (ww, hh),
                                         flags=cv2.INTER_CUBIC,
                                         borderMode=cv2.BORDER_REPLICATE)
                    print(f'  Deskewed by {angle_deg:.2f}°')
    except Exception as e:
        print(f'  Deskew skipped: {e}')

    pil_image = Image.fromarray(gray)
    return gray, pil_image


print('🖼️ Pre-processing image ...')
processed_gray, pil_processed = preprocess_image(IMAGE_PATH)

# Show side-by-side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(Image.open(IMAGE_PATH))
axes[0].set_title('Original', fontsize=13, fontweight='bold')
axes[0].axis('off')
axes[1].imshow(processed_gray, cmap='gray')
axes[1].set_title('Processed (CLAHE + Denoise + Deskew)', fontsize=13, fontweight='bold')
axes[1].axis('off')
plt.suptitle('Image Preprocessing Result', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()
print('✅ Preprocessing complete')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — OCR PIPELINE (TrOCR + EasyOCR + Tesseract)       ║
# ╚══════════════════════════════════════════════════════════════╝

# ─── TrOCR ───────────────────────────────────────────────────────────────────
def run_trocr(pil_image):
    """Microsoft Transformer OCR."""
    try:
        rgb = pil_image.convert('RGB')
        pixel_values = trocr_processor(
            images=rgb, return_tensors='pt'
        ).pixel_values.to(DEVICE)
        with torch.no_grad():
            ids = trocr_model.generate(pixel_values, max_new_tokens=512)
        text = trocr_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        conf = 0.85 if text else 0.0
        return {'text': text, 'confidence': conf, 'engine': 'TrOCR'}
    except Exception as e:
        print(f'  TrOCR error: {e}')
        return {'text': '', 'confidence': 0.0, 'engine': 'TrOCR'}


# ─── EasyOCR ─────────────────────────────────────────────────────────────────
def run_easyocr(pil_image):
    """EasyOCR multi-language OCR."""
    try:
        img_arr = np.array(pil_image.convert('RGB'))
        results = easy_reader.readtext(img_arr, detail=1)
        if not results:
            return {'text': '', 'confidence': 0.0, 'engine': 'EasyOCR', 'details': []}
        lines, confs = [], []
        for (_, text, conf) in results:
            lines.append(text)
            confs.append(conf)
        return {
            'text': '\n'.join(lines),
            'confidence': float(np.mean(confs)),
            'engine': 'EasyOCR',
            'details': [{'text': t, 'confidence': c} for (_, t, c) in results]
        }
    except Exception as e:
        print(f'  EasyOCR error: {e}')
        return {'text': '', 'confidence': 0.0, 'engine': 'EasyOCR', 'details': []}


# ─── Tesseract ───────────────────────────────────────────────────────────────
def run_tesseract(gray_array):
    """Tesseract classical OCR."""
    try:
        config = r'--oem 3 --psm 6 -l eng+hin+ben+ori'
        data = pytesseract.image_to_data(
            gray_array, config=config,
            output_type=pytesseract.Output.DICT
        )
        words, confs = [], []
        for i, word in enumerate(data['text']):
            c = int(data['conf'][i])
            if word.strip() and c > 10:
                words.append(word)
                confs.append(c)
        text = ' '.join(words)
        conf = float(np.mean(confs)) / 100.0 if confs else 0.0
        return {'text': text, 'confidence': conf, 'engine': 'Tesseract'}
    except Exception as e:
        print(f'  Tesseract error: {e}')
        return {'text': '', 'confidence': 0.0, 'engine': 'Tesseract'}


# ─── Run all three ────────────────────────────────────────────────────────────
print('Running OCR engines ...')

print('  🧠 TrOCR ...')
trocr_result = run_trocr(pil_processed)
print(f'     Confidence: {trocr_result["confidence"]:.2f}')

print('  📖 EasyOCR ...')
easyocr_result = run_easyocr(pil_processed)
print(f'     Confidence: {easyocr_result["confidence"]:.2f}')

print('  🔍 Tesseract ...')
tesseract_result = run_tesseract(processed_gray)
print(f'     Confidence: {tesseract_result["confidence"]:.2f}')

print('\n✅ All OCR engines complete')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — COMBINE OCR OUTPUTS                              ║
# ╚══════════════════════════════════════════════════════════════╝

def combine_ocr_results(trocr, easy, tess):
    """
    Combine three OCR results using confidence-weighted strategy:
    1. Sort by confidence (highest first)
    2. Use highest-confidence result as primary
    3. Supplement with unique tokens from other engines
    """
    results = sorted([trocr, easy, tess],
                     key=lambda x: x['confidence'], reverse=True)
    primary = results[0]['text'].strip()
    primary_tokens = set(primary.lower().split())

    extras = []
    for r in results[1:]:
        for token in r['text'].split():
            if token.lower() not in primary_tokens and len(token) > 2:
                extras.append(token)
                primary_tokens.add(token.lower())

    combined = primary
    if extras:
        combined += '\n' + ' '.join(extras)
    return combined.strip()


combined_text = combine_ocr_results(trocr_result, easyocr_result, tesseract_result)

# ─── Visualise confidence comparison ─────────────────────────────────────────
engines = ['TrOCR', 'EasyOCR', 'Tesseract']
confidences = [
    trocr_result['confidence'],
    easyocr_result['confidence'],
    tesseract_result['confidence'],
]
bar_colors = ['#1976d2', '#388e3c', '#f57c00']

fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.barh(engines, [c * 100 for c in confidences],
               color=bar_colors, edgecolor='white', height=0.5)
for bar, conf in zip(bars, confidences):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f'{conf * 100:.1f}%', va='center', fontsize=11, fontweight='bold')
ax.set_xlim(0, 115)
ax.set_xlabel('Confidence (%)')
ax.set_title('OCR Engine Confidence Scores', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('═' * 60)
print('📄 COMBINED OCR TEXT:')
print('═' * 60)
print(combined_text)
print('═' * 60)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8 — LANGUAGE DETECTION & TRANSLATION                 ║
# ╚══════════════════════════════════════════════════════════════╝

LANG_MAP = {
    'hi': 'Hindi', 'bn': 'Bengali', 'or': 'Odia',
    'ta': 'Tamil', 'te': 'Telugu', 'mr': 'Marathi',
    'gu': 'Gujarati', 'pa': 'Punjabi', 'ur': 'Urdu',
    'kn': 'Kannada', 'ml': 'Malayalam', 'en': 'English',
}


def detect_and_translate(text):
    """Detect language and translate to English if needed."""
    if not text.strip():
        return {'original': text, 'translated': text,
                'language': 'unknown', 'lang_code': 'en'}
    try:
        lang_code = detect(text)
    except LangDetectException:
        lang_code = 'en'

    lang_name = LANG_MAP.get(lang_code, lang_code.upper())

    if lang_code == 'en':
        return {'original': text, 'translated': text,
                'language': 'English', 'lang_code': 'en'}

    try:
        translated = GoogleTranslator(source='auto', target='en').translate(text)
    except Exception as e:
        print(f'  Translation warning: {e}')
        translated = text

    return {
        'original': text,
        'translated': translated,
        'language': lang_name,
        'lang_code': lang_code,
    }


print('🌐 Detecting language and translating ...')
translation_result = detect_and_translate(combined_text)

print(f'\n  Detected Language : {translation_result["language"]}')
print(f'  Language Code     : {translation_result["lang_code"]}')
if translation_result['lang_code'] != 'en':
    print('\n  📘 English Translation:')
    print('  ' + translation_result['translated'])
else:
    print('  ℹ️ Text is already in English — no translation needed.')

# Use translated text for downstream NLP
work_text = translation_result['translated'] or combined_text
print('\n✅ Translation step complete')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — MEDICAL INFORMATION EXTRACTION (RULE-BASED)      ║
# ╚══════════════════════════════════════════════════════════════╝

MEDICINE_PATTERNS = [
    r'\b(?:Tab|Cap|Syp|Inj|Oint|Drop|Syr)\b\.?\s*\w+',
    r'\b\w+(?:cin|mycin|cillin|zole|pril|sartan|olol|statin|pam|oxacin|mab)\b',
    r'\b\w+\s+\d+\s*(?:mg|mcg|ml|g|IU)\b',
    r'\b(?:Amoxicillin|Paracetamol|Metformin|Aspirin|Ibuprofen|Azithromycin|'
    r'Ciprofloxacin|Omeprazole|Atorvastatin|Metoprolol|Amlodipine|'
    r'Cetirizine|Pantoprazole|Dolo|Calpol|Augmentin|Crocin)\b',
]
DOSAGE_PATTERNS = [
    r'\b\d+\s*(?:mg|mcg|ml|g|IU)\b',
    r'\b(?:once|twice|thrice|OD|BD|TDS|QID|PRN|SOS|HS|AC|PC)\b',
    r'\b\d+-\d+-\d+\b',
    r'\b\d+\s*tab(?:let)?s?\b',
    r'\b\d+\s*times?\s*(?:a\s*)?day\b',
    r'\bfor\s+\d+\s*days?\b',
]
DIAGNOSIS_PATTERNS = [
    r'(?:Diagnosis|Dx|Impression|Findings?|Assessment)\s*:?\s*(.+)',
    r'\b(?:fever|hypertension|diabetes|infection|pneumonia|bronchitis|asthma|'
    r'anaemia|anemia|UTI|URTI|cold|cough|flu|malaria|typhoid|dengue|gastritis|'
    r'migraine|arthritis|fracture|allergy|pharyngitis|tonsillitis)\b',
]


def extract_medical_info(text):
    """Rule-based extraction of medical entities."""
    info = {'Patient': '', 'DOB': '', 'Date': '', 'Doctor': '',
            'Hospital': '', 'Medicines': [], 'Dosages': [], 'Diagnosis': []}

    # Patient
    m = re.search(r'(?:Patient|Name|Pt\.?)\s*[:\-]?\s*([A-Z][a-zA-Z\s\.]{2,40})', text)
    if m: info['Patient'] = m.group(1).strip()

    # DOB
    m = re.search(r'(?:DOB|Date\s*of\s*Birth)\s*[:\-]?\s*(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})', text)
    if m: info['DOB'] = m.group(1).strip()

    # Date
    m = re.search(r'(?:Date|Dt\.?)\s*[:\-]?\s*(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})', text)
    if m: info['Date'] = m.group(1).strip()

    # Doctor
    m = re.search(r'(?:Dr\.?|Doctor|Physician)\s*[:\-]?\s*([A-Z][a-zA-Z\s\.]{2,40})', text)
    if m: info['Doctor'] = m.group(1).strip()

    # Medicines
    meds = set()
    for pat in MEDICINE_PATTERNS:
        for hit in re.finditer(pat, text, re.IGNORECASE):
            t = hit.group(0).strip()
            if len(t) > 2: meds.add(t)
    info['Medicines'] = sorted(meds)

    # Dosages
    doses = set()
    for pat in DOSAGE_PATTERNS:
        for hit in re.finditer(pat, text, re.IGNORECASE):
            doses.add(hit.group(0).strip())
    info['Dosages'] = sorted(doses)

    # Diagnosis
    diag = set()
    for pat in DIAGNOSIS_PATTERNS:
        for hit in re.finditer(pat, text, re.IGNORECASE):
            d = hit.group(0).strip()
            if len(d) > 2: diag.add(d)
    info['Diagnosis'] = sorted(diag)
    return info


print('💊 Extracting medical information ...')
medical_info = extract_medical_info(work_text)

print('\n📋 STRUCTURED MEDICAL DATA:')
print('─' * 50)
for key, val in medical_info.items():
    if isinstance(val, list):
        print(f'  {key:15}: {len(val)} item(s)')
        for item in val:
            print(f'    • {item}')
    else:
        print(f'  {key:15}: {val or "—"}')
print('─' * 50)
print('✅ Medical extraction complete')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — BERT NER EXTRACTION + ENTITY HIGHLIGHTING       ║
# ╚══════════════════════════════════════════════════════════════╝

def extract_entities_bert(text):
    """Run BERT NER and return grouped entities."""
    if not text.strip():
        return {}
    try:
        raw = ner_pipeline(text[:512])
    except Exception as e:
        print(f'  BERT NER error: {e}')
        return {}
    grouped = {}
    for ent in raw:
        label = ent.get('entity_group', ent.get('entity', 'MISC'))
        word = ent.get('word', '').strip()
        score = round(float(ent.get('score', 0.0)), 3)
        if not word or len(word) < 2:
            continue
        if label not in grouped:
            grouped[label] = []
        grouped[label].append({'word': word, 'score': score})
    return grouped


def highlight_entities_console(text, bert_entities):
    """Print text with highlighted entity markers."""
    COLOR_MAP = {
        'PER': '\033[1;32m',   # Green
        'ORG': '\033[1;34m',   # Blue
        'LOC': '\033[1;33m',   # Yellow
        'MISC': '\033[1;35m',  # Magenta
    }
    RESET = '\033[0m'
    highlighted = text
    for label, items in bert_entities.items():
        color = COLOR_MAP.get(label, '\033[1;36m')
        for item in items:
            word = item['word']
            highlighted = highlighted.replace(
                word, f"{color}[{label}:{word}]{RESET}"
            )
    return highlighted


print('🤖 Running BERT NER ...')
bert_entities = extract_entities_bert(work_text)

print('\n🏷️ NAMED ENTITIES FOUND:')
print('─' * 50)
if bert_entities:
    for label, items in bert_entities.items():
        words = ', '.join([f'{i["word"]} ({i["score"]:.2f})' for i in items])
        print(f'  [{label}] {words}')
else:
    print('  No entities extracted.')
print('─' * 50)

# Highlighted text
print('\n🎨 Highlighted Text:')
print(highlight_entities_console(work_text[:500], bert_entities))
print('\n✅ BERT NER complete')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — PDF REPORT GENERATION                           ║
# ╚══════════════════════════════════════════════════════════════╝

def generate_pdf_report(
    ocr_text, translated_text, detected_language,
    med_info, bert_ents,
    trocr_conf, easyocr_conf, tesseract_conf,
    output_path='final_medical_report.pdf'
):
    doc = SimpleDocTemplate(
        output_path, pagesize=A4,
        rightMargin=2*cm, leftMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()
    story = []

    # ── Styles ────────────────────────────────────────────────────────────────
    title_s = ParagraphStyle('T', parent=styles['Title'], fontSize=17,
                              textColor=colors.HexColor('#1a237e'), alignment=TA_CENTER)
    sub_s   = ParagraphStyle('S', parent=styles['Normal'], fontSize=9,
                              textColor=colors.HexColor('#546e7a'), alignment=TA_CENTER, spaceAfter=10)
    sec_s   = ParagraphStyle('H', parent=styles['Heading2'], fontSize=11,
                              textColor=colors.HexColor('#0d47a1'), spaceBefore=10, spaceAfter=4)
    body_s  = ParagraphStyle('B', parent=styles['Normal'], fontSize=9, leading=14)
    foot_s  = ParagraphStyle('F', parent=styles['Normal'], fontSize=7,
                              textColor=colors.grey, alignment=TA_CENTER)

    def safe(t): return (t or '').replace('&','&amp;').replace('<','&lt;').replace('>','&gt;')

    grid = TableStyle([
        ('FONTNAME', (0,0),(-1,0), 'Helvetica-Bold'),
        ('FONTSIZE', (0,0),(-1,-1), 9),
        ('GRID', (0,0),(-1,-1), 0.5, colors.HexColor('#bdbdbd')),
        ('PADDING', (0,0),(-1,-1), 5),
        ('ROWBACKGROUNDS', (0,1),(-1,-1), [colors.white, colors.HexColor('#f5f5f5')]),
    ])

    # ── Header ────────────────────────────────────────────────────────────────
    story.append(Paragraph('🏥 Multilingual Medical OCR Report', title_s))
    story.append(Paragraph(
        f'Generated: {datetime.now().strftime("%d %B %Y, %H:%M")} | '
        f'Language: <b>{detected_language}</b>', sub_s))
    story.append(HRFlowable(width='100%', thickness=2, color=colors.HexColor('#1a237e')))
    story.append(Spacer(1, 0.3*cm))

    # ── Patient Details ───────────────────────────────────────────────────────
    story.append(Paragraph('Patient & Prescription Details', sec_s))
    pt_data = [
        ['Field', 'Value'],
        ['Patient Name', safe(med_info.get('Patient')) or '—'],
        ['Date of Birth', safe(med_info.get('DOB')) or '—'],
        ['Prescription Date', safe(med_info.get('Date')) or '—'],
        ['Doctor', safe(med_info.get('Doctor')) or '—'],
        ['Hospital', safe(med_info.get('Hospital')) or '—'],
    ]
    t = Table(pt_data, colWidths=[5*cm, 12*cm])
    t.setStyle(grid)
    t.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#1a237e')),
                            ('TEXTCOLOR',(0,0),(-1,0),colors.white)]))
    story += [t, Spacer(1, 0.3*cm)]

    # ── Confidence ────────────────────────────────────────────────────────────
    story.append(Paragraph('OCR Confidence Scores', sec_s))
    conf_data = [
        ['Engine', 'Confidence', 'Status'],
        ['TrOCR', f'{trocr_conf*100:.1f}%', '✓ Good' if trocr_conf>0.6 else '⚠ Low'],
        ['EasyOCR', f'{easyocr_conf*100:.1f}%', '✓ Good' if easyocr_conf>0.6 else '⚠ Low'],
        ['Tesseract', f'{tesseract_conf*100:.1f}%', '✓ Good' if tesseract_conf>0.6 else '⚠ Low'],
    ]
    ct = Table(conf_data, colWidths=[5*cm, 5*cm, 7*cm])
    ct.setStyle(grid)
    ct.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#0d47a1')),
                             ('TEXTCOLOR',(0,0),(-1,0),colors.white)]))
    story += [ct, Spacer(1, 0.3*cm)]

    # ── OCR Text ──────────────────────────────────────────────────────────────
    story.append(Paragraph('Extracted OCR Text (Combined)', sec_s))
    story.append(Paragraph(safe(ocr_text) or 'No text extracted.', body_s))
    story.append(Spacer(1, 0.2*cm))

    # ── Translation ───────────────────────────────────────────────────────────
    if detected_language != 'English':
        story.append(Paragraph(f'English Translation (from {detected_language})', sec_s))
        story.append(Paragraph(safe(translated_text) or '—', body_s))
        story.append(Spacer(1, 0.2*cm))

    # ── Medicines ─────────────────────────────────────────────────────────────
    story.append(Paragraph('Medicines & Dosages', sec_s))
    meds = med_info.get('Medicines', [])
    doses = med_info.get('Dosages', [])
    if meds or doses:
        med_rows = [['#', 'Medicine', 'Dosage']]
        for i in range(max(len(meds), len(doses), 1)):
            med_rows.append([
                str(i+1),
                meds[i] if i < len(meds) else '—',
                doses[i] if i < len(doses) else '—',
            ])
        mt = Table(med_rows, colWidths=[1*cm, 8*cm, 8*cm])
        mt.setStyle(grid)
        mt.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#2e7d32')),
                                 ('TEXTCOLOR',(0,0),(-1,0),colors.white)]))
        story.append(mt)
    else:
        story.append(Paragraph('No medicines identified.', body_s))
    story.append(Spacer(1, 0.3*cm))

    # ── Diagnosis ─────────────────────────────────────────────────────────────
    story.append(Paragraph('Diagnosis / Clinical Findings', sec_s))
    diags = med_info.get('Diagnosis', [])
    if diags:
        for dx in diags:
            story.append(Paragraph(f'• {safe(dx)}', body_s))
    else:
        story.append(Paragraph('No diagnosis keywords found.', body_s))
    story.append(Spacer(1, 0.3*cm))

    # ── BERT NER ──────────────────────────────────────────────────────────────
    story.append(Paragraph('BERT Named Entity Recognition', sec_s))
    if bert_ents:
        ner_rows = [['Type', 'Entities', 'Avg Conf']]
        for lbl, items in bert_ents.items():
            words = ', '.join([i['word'] for i in items])
            avg   = float(np.mean([i['score'] for i in items]))
            ner_rows.append([lbl, words, f'{avg:.3f}'])
        nt = Table(ner_rows, colWidths=[3*cm, 12*cm, 2*cm])
        nt.setStyle(grid)
        nt.setStyle(TableStyle([('BACKGROUND',(0,0),(-1,0),colors.HexColor('#4a148c')),
                                 ('TEXTCOLOR',(0,0),(-1,0),colors.white)]))
        story.append(nt)
    else:
        story.append(Paragraph('No entities found.', body_s))

    # ── Footer ────────────────────────────────────────────────────────────────
    story += [
        Spacer(1, 0.5*cm),
        HRFlowable(width='100%', thickness=1, color=colors.HexColor('#90a4ae')),
        Paragraph(
            '<i>Multilingual Medical OCR System | '
            'For informational purposes only. Not a substitute for professional medical advice.</i>',
            foot_s
        ),
    ]

    doc.build(story)
    return output_path


print('📄 Generating PDF report ...')
PDF_PATH = 'final_medical_report.pdf'
generate_pdf_report(
    ocr_text=combined_text,
    translated_text=translation_result['translated'],
    detected_language=translation_result['language'],
    med_info=medical_info,
    bert_ents=bert_entities,
    trocr_conf=trocr_result['confidence'],
    easyocr_conf=easyocr_result['confidence'],
    tesseract_conf=tesseract_result['confidence'],
    output_path=PDF_PATH,
)
print(f'✅ PDF saved: {PDF_PATH}')

if IN_COLAB:
    print('⬇️ Downloading PDF ...')
    colab_files.download(PDF_PATH)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — FULL SELF-CONTAINED PIPELINE FUNCTION           ║
# ╚══════════════════════════════════════════════════════════════╝

def run_medical_ocr_pipeline(image_path, output_pdf='medical_report.pdf', verbose=True):
    """
    ✅ Complete end-to-end Medical OCR Pipeline.

    Parameters
    ----------
    image_path  : str   — path to medical image
    output_pdf  : str   — output PDF filename
    verbose     : bool  — print progress

    Returns
    -------
    dict with keys:
        combined_text, translation, medical_info,
        bert_entities, pdf_path, elapsed_seconds
    """
    t0 = time.time()
    if verbose: print(f'\n🚀 Medical OCR Pipeline started: {image_path}\n')

    # 1. Preprocess
    gray, pil = preprocess_image(image_path)
    if verbose: print('  [1/7] ✅ Preprocessed')

    # 2. OCR
    tr = run_trocr(pil)
    er = run_easyocr(pil)
    ts = run_tesseract(gray)
    if verbose: print(f'  [2/7] ✅ OCR | TrOCR:{tr["confidence"]:.2f}  '
                      f'EasyOCR:{er["confidence"]:.2f}  '
                      f'Tesseract:{ts["confidence"]:.2f}')

    # 3. Combine
    combined = combine_ocr_results(tr, er, ts)
    if verbose: print(f'  [3/7] ✅ Combined ({len(combined)} chars)')

    # 4. Translate
    trans = detect_and_translate(combined)
    if verbose: print(f'  [4/7] ✅ Language: {trans["language"]}')

    # 5. Medical extraction
    wt = trans['translated'] or combined
    minfo = extract_medical_info(wt)
    if verbose: print(f'  [5/7] ✅ Medicines: {len(minfo["Medicines"])} | '
                      f'Diagnoses: {len(minfo["Diagnosis"])}')

    # 6. BERT NER
    berts = extract_entities_bert(wt)
    if verbose: print(f'  [6/7] ✅ BERT entities: {list(berts.keys())}')

    # 7. PDF
    pdf = generate_pdf_report(
        ocr_text=combined, translated_text=trans['translated'],
        detected_language=trans['language'], med_info=minfo,
        bert_ents=berts, trocr_conf=tr['confidence'],
        easyocr_conf=er['confidence'], tesseract_conf=ts['confidence'],
        output_path=output_pdf,
    )
    elapsed = round(time.time() - t0, 2)
    if verbose: print(f'  [7/7] ✅ PDF generated: {pdf}')
    if verbose: print(f'\n🎉 Pipeline complete in {elapsed}s!')

    # Download in Colab
    if IN_COLAB and os.path.exists(pdf):
        colab_files.download(pdf)

    return {
        'combined_text': combined, 'translation': trans,
        'medical_info': minfo, 'bert_entities': berts,
        'pdf_path': pdf, 'elapsed_seconds': elapsed,
        'confidence': {
            'trocr': tr['confidence'],
            'easyocr': er['confidence'],
            'tesseract': ts['confidence'],
        },
    }


# ─── Run the full pipeline on the uploaded image ──────────────────────────────
print('=' * 65)
print('  FULL PIPELINE DEMO  ')
print('=' * 65)

results = run_medical_ocr_pipeline(IMAGE_PATH, output_pdf='final_medical_report.pdf')

print('\n📊 SUMMARY:')
print(f'  Language      : {results["translation"]["language"]}')
print(f'  Characters    : {len(results["combined_text"])}')
print(f'  Medicines     : {len(results["medical_info"]["Medicines"])}')
print(f'  BERT entities : {sum(len(v) for v in results["bert_entities"].values())}')
print(f'  Time          : {results["elapsed_seconds"]}s')
print(f'  PDF           : {results["pdf_path"]}')
print('\n✅ System ready for production use!')